In [8]:
import scanpy as sc
import pandas as pd
from scipy.sparse import csr_matrix

In [ ]:
print("1. Loading the massive matrix... (Your M4 will chew through this quickly)")
# The .T is critical! R stores data as (Genes x Cells). Python needs it as (Cells x Genes).
gli3_ko = sc.read_mtx("/Users/christianlangridge/Desktop/SMT-Pipeline/data/GLI3 KO validation/data_matrices/counts.mtx.gz").T 

# Convert to a Compressed Sparse Row (CSR) matrix. 
# This makes future operations (like UMAPs and filtering) lightning fast.
gli3_ko.X = csr_matrix(gli3_ko.X)


In [ ]:

print("2. Loading labels and metadata...")
# Use sep='\t' since we saved them as .tsv (tab-separated) in Kaggle
genes = pd.read_csv("/Users/christianlangridge/Desktop/SMT-Pipeline/data/GLI3 KO validation/data_matrices/features.tsv.gz", header=None, sep='\t')
barcodes = pd.read_csv("/Users/christianlangridge/Desktop/SMT-Pipeline/data/GLI3 KO validation/data_matrices/barcodes.tsv.gz", header=None, sep='\t')
metadata = pd.read_csv("/Users/christianlangridge/Desktop/SMT-Pipeline/data/GLI3 KO validation/data_matrices/meta.tsv.gz", index_col=0)

print("3. Stitching it all together...")
# Assign the labels to the AnnData object
gli3_ko.var_names = genes[0].values
gli3_ko.obs_names = barcodes[0].values
gli3_ko.obs = metadata


In [ ]:
### SAVING STICHED H5AD OBJECT 

print("4. Saving your Master File...")
# Save it as a compressed HDF5 file. This is your new gold standard.
gli3_ko.write("GLI3_ko_raw.h5ad", compression="gzip")

print("-" * 30)
print("✅ STITCHING COMPLETE!")
print(f"Total Cells: {gli3_ko.n_obs}")
print(f"Total Genes: {gli3_ko.n_vars}")
print("Your data is ready for analysis!")

In [ ]:
wls_ko = sc.read_h5ad("/Users/christianlangridge/Desktop/SMT-Pipeline/data/WLS KO validation/WLS_ko.h5ad")
print(wls_ko)

In [ ]:
training_data = sc.read_h5ad("/Users/christianlangridge/Desktop/SMT-Pipeline/data/Training data/AnnData object/neurectoderm_complete.h5ad")
print(training_data)

In [9]:
gli3_ko = sc.read_h5ad("/Users/christianlangridge/Desktop/SMT-Pipeline/data/GLI3 KO validation/GLI3_ko.h5ad")
print(gli3_ko)

AnnData object with n_obs × n_vars = 49718 × 33538
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent_mito', 'RNA_snn_res.0.8', 'seurat_clusters', 'organoid', 'line', 'p_singlet', 'age', 'nowakowski_prediction', 'RNA_snn_res.1', 'glyc_cluster'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'


In [10]:
# 1. Check the main matrix (X)
# If these are integers (e.g., 0, 1, 5, 20), they are raw counts.
# If they are floats (e.g., 0.12, 1.5), they are normalized.
print(gli3_ko.X[:5, :5])

# 2. Check the 'raw' attribute (Most common for published data)
if gli3_ko.raw is not None:
    raw_counts = gli3_ko.raw.X
    print("Found raw counts in gli3_ko.raw")
else:
    print("No gli3_ko.raw found.")

# 3. Check for a 'counts' layer
if 'counts' in gli3_ko.layers:
    raw_counts = gli3_ko.layers['counts']
    print("Found raw counts in gli3_ko.layers['counts']")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 0 stored elements and shape (5, 5)>
Found raw counts in gli3_ko.raw


In [11]:
# Convert the raw slot back into its own AnnData object
gli3_ko_raw_generated = gli3_ko.raw.to_adata()

# Quick check to ensure there is actually data in there
print(f"Total counts in matrix: {gli3_ko_raw_generated.X.sum()}")

Total counts in matrix: 703213316.0


In [12]:
gli3_ko_raw_generated.obs.head()

,orig.ident,nCount_RNA,nFeature_RNA,percent_mito,RNA_snn_res.0.8,seurat_clusters,organoid,line,p_singlet,age,nowakowski_prediction,RNA_snn_res.1,glyc_cluster
RNA_DAY11_SJ_AAACCCAAGTCGTTAC-1,RNA_DAY11_SJ,13722.0,4238,0.040154,3,8,DAY11_H9,H9,1.0,11.0,IPC,8,0
RNA_DAY11_SJ_AAACCCAGTAGCGAGT-1,RNA_DAY11_SJ,34067.0,6444,0.094226,3,16,DAY11_Wibj2,Wibj2,1.0,11.0,IPC,16,0
RNA_DAY11_SJ_AAACCCAGTGACTGAG-1,RNA_DAY11_SJ,31958.0,6389,0.051067,3,19,DAY11_Wibj2,Wibj2,1.0,11.0,IPC,19,0
RNA_DAY11_SJ_AAACGAAAGAAGCCAC-1,RNA_DAY11_SJ,22512.0,5511,0.063699,3,8,DAY11_H9,H9,1.0,11.0,RG,8,0
RNA_DAY11_SJ_AAACGAATCTGCTGAA-1,RNA_DAY11_SJ,23513.0,5486,0.074895,3,8,DAY11_Wibj2,Wibj2,1.0,11.0,IPC,8,0


In [ ]:
gli3_ko_raw = sc.read_h5ad("/Users/christianlangridge/Desktop/SMT-Pipeline/path/data_preparation/GLI3_ko_raw.h5ad")
print(gli3_ko_raw)

In [ ]:
# A quick standard pipeline to generate a UMAP in Python:
sc.pp.normalize_total(training_data, target_sum=1e4)
sc.pp.log1p(training_data)
sc.pp.highly_variable_genes(training_data, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.tl.pca(training_data, svd_solver='arpack')
sc.pp.neighbors(training_data, n_neighbors=10, n_pcs=40)
sc.tl.umap(training_data)

# Convert the cluster column from numbers to categories
training_data.obs['seurat_clusters'] = training_data.obs['seurat_clusters'].astype('category')



In [ ]:
# Now you can plot it using the metadata columns we saved!
sc.pl.umap(training_data, color=['class3', 'seurat_clusters'], wspace=0.5)

In [ ]:
print(gli3_ko.obs.columns.tolist())

In [ ]:
print(gli3_ko_raw.obs.columns.tolist())

In [ ]:
gli3_ko_raw.obs.head()

In [ ]:
training_data.obs.head()

In [ ]:
GLS_ko_AnnData.obs.head()

In [ ]:
# A quick standard pipeline to generate a UMAP in Python:
sc.pp.normalize_total(GLS_ko_AnnData, target_sum=1e4)
sc.pp.log1p(GLS_ko_AnnData)
sc.pp.highly_variable_genes(GLS_ko_AnnData, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.tl.pca(GLS_ko_AnnData, svd_solver='arpack')
sc.pp.neighbors(GLS_ko_AnnData, n_neighbors=10, n_pcs=40)
sc.tl.umap(GLS_ko_AnnData)

# Convert the cluster column from numbers to categories
GLS_ko_AnnData.obs['seurat_clusters'] = GLS_ko_AnnData.obs['seurat_clusters'].astype('category')

In [ ]:
# Now you can plot it using the metadata columns we saved!
sc.pl.umap(GLS_ko_AnnData, color=['class3', 'seurat_clusters'], wspace=0.5)